In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc 
import anndata as ad
import h5py 
import glob
import matplotlib.pyplot as plt
import glob
import os

In [ ]:
base_path = '/home/EOCRC_atlas/'

In [ ]:
# load epithelial adata 
adata = sc.read_h5ad(os.path.join(base_path, "data/yocrc_Epithelial_annotation.h5ad"))

In [ ]:
# plot only major cell types in specified order
annot_order = [
    "LGR5 stem cell-like",
    "MT-Ribo-hi epithelial",
    "CEACAM1 colonocyte-like",
    "MUC2 goblet-like",
    "Enteroendocrine-like",
]
adata = adata[adata.obs['Annotation_Tier2'].isin(annot_order)]
adata.obs["Annotation_Tier2"] = pd.Categorical(adata.obs["Annotation_Tier2"], categories=annot_order, ordered=True)
adata.obs["Annotation_Tier2"] = adata.obs["Annotation_Tier2"].cat.remove_unused_categories()

In [ ]:
# score and visualize Kong et al signatures (non-malignant colon signatures) 
lk_markers = {
    'Enterochromaffin' : ['CHGA', 'TPH1', 'CES1',  'SLC38A11', 'RAB3C'],
    'best4' : ['BEST4', 'CA7', 'CA4', 'SPIB', 'OTOP2'],#NOTCH2NL 
    'ca1_ca2_ca4neg' : ['CA1', 'SLC26A2', 'CA2', 'SLC26A3', 'KRT19', 'SELENBP1', 'PKIB', 'UGT2B17', 'CES2'], 
    'tmigd1_mep1a' : ['TMIGD1', 'MEP1A', 'APOA4', 'APOC3', 'APOA1', 'FABP6'],
    'tmigd1_mep1a_gsta1' : ['GSTA1', 'GSTA2', 'TMIGD1', 'MEP1A'], 
    'enteroendocrine' : ['PCSK1N', 'PYY', 'CHGA', 'GCG', 'CRYBA2', 'SCGN', 'FEV', 'SCG5', 'INSL5', 'MS4A8'], 
    'hbb_hba' : ['HBB', 'HBA2', 'HBA1'], 
    'mettl12_mafb' : ['MAFB'],#METTL12 
    'cycling' : ['UBE2C', 'PTTG1', 'HMGB2', 'TOP2A', 'CKS2', 'CENPW', 'CDKN3', 'STMN1', 'TUBB4B', 'HIST1H4C'], 
    'goblet_tff1neg' : [ 'MUC2', 'RETNLB', 'SPINK4', 'ITLN1', 'CLCA1', 'FCGBP', 'TFF3', 'ST6GALNAC1', 'LRRC26', 'REP15'], 
    'goblet_tff1pos' : ['MUC2', 'SPINK4', 'FCGBP', 'CLCA1', 'ZG16', 'TFF1', 'BCAS1', 'CEACAM5'], 
    'goblet_spink4' : ['SPINK4', 'MUC2', 'FCGBP', 'CLCA1', 'ITLN1', 'TFF3', 'TFF1', 'S100P', 'RETNLB', 'LRRC26'], 
    'l_cells' :  ['CHGA', 'NTS', 'PYY', 'GCG', 'CCK'], 
    'paneth' :  ['DEFA5', 'DEFA6', 'REG3A', 'PRSS1', 'ITLN2', 'PLA2G2A'], 
    'stem_olmfa' :  ['OLFM4', 'REG1A'], 
    'stem_olmfa_gsta1' :  ['FABP1', 'GSTA1', 'AKR1C3', 'KRT19', 'MAOA', 'CES2', 'CBR1', 'RBP2', 'PTGR1', 'LIMA1'], 
    'stem_olmfa_lgr5' :  ['LGR5', 'OLFM4'], 
    'stem_olmfa_pcna' :  ['PCNA', 'RANBP1', 'OLFM4', 'DUT', 'SIVA1'], #STRA13
    'tuft' :  ['SH2D6', 'LRMP', 'AVIL', 'BMX', 'AZGP1', 'MATK', 'TRPM5']} #7SK_ENSG00000260682
    
for name, markers in lk_markers.items():
    sc.tl.score_genes(adata, markers, score_name = name, use_raw=True)
sc.pl.umap(adata, color=list(lk_markers.keys()), size=4, cmap='inferno')

In [ ]:
# create dot plot comparing kong signatures to our annotated atlas - exclude patient specific and mixed 
cell_type_mask = ~adata.obs['Annotation_Tier2'].isin(['Patient-specific', 'Mixed - epithelial'])
adata_filtered = adata[cell_type_mask].copy()


kong_order = [
    "stem_olmfa_lgr5",
    "stem_olmfa_pcna",
    "stem_olmfa",
    "cycling",
    "stem_olmfa_gsta1",
    "ca1_ca2_ca4neg",
    "tmigd1_mep1a",
    "tmigd1_mep1a_gsta1",
    "best4",
    "goblet_spink4",
    "goblet_tff1neg",
    "goblet_tff1pos",
    "paneth",
    "enteroendocrine",
    "Enterochromaffin",
    "l_cells",
    "tuft",
    "mettl12_mafb",
    "hbb_hba",
]

# make dot plot 
sc.pl.dotplot(
    adata_filtered, 
    var_names=kong_order, 
    groupby='Annotation_Tier2', 
    standard_scale='var', 
    dendrogram=False,      
    swap_axes=True, 
    show=False
)

plt.savefig(os.path.join(base_path, f'results/annotConfirm/kong_yocrc_dotplot.pdf'), bbox_inches='tight', dpi=600)
plt.show()

In [ ]:
# score and visualize Pelka et al annotation signatures
df = pd.read_csv(os.path.join(base_path, 'docs/crc_epi_signatures.csv'), keep_default_na=False)
subsets = df.columns
for s in subsets:
    sc.tl.score_genes(adata, df[s].values[1:df.shape[0]], score_name = s, use_raw=True)
sc.pl.umap(adata, color=subsets, size=2, cmap='inferno')

In [ ]:
# create dot plot comparing pelka signatures to our annotated atlas - exclude patient specific and mixed 
cell_type_mask = ~adata.obs['Annotation_Tier2'].isin(['Patient-specific', 'Mixed - epithelial'])
adata_filtered = adata[cell_type_mask].copy()

pelka_order = [
    "cE01_Stem_TA-like",
    "cE03_Stem_TA-like_prolif",
    "cE04_Enterocyte1",
    "cE05_Enterocyte2",
    "cE09_Best4",
    "cE02_Stem_TA-like_ImmatureGoblet",
    "cE06_ImmatureGoblet",
    "cE08_Goblet",
    "cE07_Goblet_Enterocyte",
    "cE11_Enteroendocrine",
    "cE10_Tuft",
]

# make dot plot 
sc.pl.dotplot(
    adata_filtered, 
    var_names=pelka_order, 
    groupby='Annotation_Tier2', 
    standard_scale='var', 
    dendrogram=False,      
    swap_axes=True, 
    show=False
)

plt.savefig(os.path.join(base_path, f'results/annotConfirm/pelka_yocrc_dotplot.pdf'), bbox_inches='tight', dpi=600)
plt.show()

In [ ]:
# load becker signatures to score and visualize against our atlas 
df_becker = pd.read_csv(os.path.join(base_path, 'results/YOCRC_Becker/calcualted_cell_type_markers.csv'), keep_default_na=False)
df_becker = df_becker.rename(columns={'Unnamed: 0': 'Gene'})
becker_wide = (
    df_becker.groupby('cluster')['Gene']
    .apply(list)
    .apply(pd.Series)
    .T
)
becker_wide.index.name = None
becker_wide

In [ ]:
# score becker signatures 
subsets = becker_wide.columns
for s in subsets:
    sc.tl.score_genes(adata, becker_wide[s].values[1:becker_wide.shape[0]], score_name = s, use_raw=True)

sc.pl.umap(adata, color=subsets, size=2, cmap='inferno')

In [ ]:
# create dot plot comparing becker signatures to our annotated atlas - exclude patient specific and mixed 
cell_type_mask = ~adata.obs['Annotation_Tier2'].isin(['Patient-specific', 'Mixed - epithelial'])
adata_filtered = adata[cell_type_mask].copy()

becker_order = [
    "Stem",
    "CyclingTA",
    "TA1",
    "TA2", 
    "Immature Enterocytes",
    "Enterocyte Progenitors",
    "Enterocytes",
    "Best4+ Enterocytes",
    "Immature Goblet",
    "Goblet",
    "Enteroendocrine",
    "Tuft",
]

# make dot plots
sc.pl.dotplot(
    adata_filtered, 
    var_names=becker_order, 
    groupby='Annotation_Tier2', 
    standard_scale='var', 
    dendrogram=False,      
    swap_axes=True, 
    show=False
)

plt.savefig(os.path.join(base_path, f'results/annotConfirm/becker_yocrc_dotplot.pdf'), bbox_inches='tight', dpi=600)
plt.show()